# P9_GNN_Temporal_Multiseed

Evaluación temporal de **GCN, GAT, GraphSAGE y GIN** con 20 semillas.

- Entrenamiento/validación: grafo histórico 2000–2020; etiquetas 2021–2022.
- Test final: grafo histórico 2000–2022; etiquetas 2023–2024.
- Las aristas objetivo futuras no entran al *message passing*.
- `RUN_FINAL_TEST=False` por defecto para mantener cerrado el test.

**Dependencias:** `torch`, `torch-geometric`, `networkx`, `pandas`, `scikit-learn`.


In [1]:
#!pip install torch torch-geometric

In [2]:
import sys
print(sys.executable)

C:\Users\pilarang\envs\EnvGraphAcademicos\python.exe


In [3]:
#import sys
#!{sys.executable} -m pip uninstall -y torch torchvision torchaudio

In [4]:
#!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [5]:
#import torch
#print(torch.__version__)
#print(torch.cuda.is_available())

In [6]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import copy
import json
import random
import warnings

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score
)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
except ImportError as exc:
    raise ImportError(
        "P9 requiere PyTorch y PyTorch Geometric. En el ambiente de Jupyter ejecute, si hace falta:\n"
        "    pip install torch torch-geometric\n"
        "y reinicie el kernel."
    ) from exc

warnings.filterwarnings('ignore', category=RuntimeWarning)

# =============================================================================
# P9. GNN temporal con multiples semillas
# =============================================================================
# Objetivo:
#   Evaluar GCN, GAT, GraphSAGE y GIN bajo el protocolo temporal corregido.
#
# Protocolo:
#   - Grafo para entrenamiento/validacion: historia 2000-2020.
#   - Etiquetas de desarrollo: nuevas colaboraciones 2021-2022 (P4).
#   - Grafo para test: historia 2000-2022.
#   - Etiquetas de test: nuevas colaboraciones 2023-2024 (P4).
#   - Ninguna arista objetivo futura entra al message passing del snapshot que la predice.
#   - 20 semillas de inicializacion/optimizacion por arquitectura.
#
# Estrategia de entrenamiento final por semilla:
#   1) train P4 (27) + validacion P4 (7) para seleccionar best_epoch por early stopping.
#   2) reinicializar y entrenar con development completo (34), en el grafo 2000-2020,
#      durante best_epoch.
#   3) si RUN_FINAL_TEST=True, aplicar pesos aprendidos al snapshot historico 2000-2022
#      y evaluar una sola vez el conjunto 2023-2024 (112 pares).
#
# IMPORTANTE:
#   RUN_FINAL_TEST=False por defecto. Primero revisar resultados de validacion.
# =============================================================================

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
OUT_DIR = BASE_DIR / 'P9_gnn_temporal_multiseed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_FINAL_TEST = True
N_SEEDS = 32
SEEDS = list(range(41, 41 + N_SEEDS))
THRESHOLD = 1

# Hiperparametros FIJOS para evitar seleccionar configuracion usando el test.
HIDDEN_DIM = 32
EMBED_DIM = 16
DROPOUT = 0.20
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 400
PATIENCE = 50
MIN_DELTA = 1e-5

MODEL_NAMES = ['GCN', 'GAT', 'GraphSAGE', 'GIN']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FILES = {
    'full_2000_2020': BASE_DIR / 'BD_UNAM_Graph_2000_2020.gml',
    'full_2021_2022': BASE_DIR / 'BD_UNAM_Graph_2021_2022.gml',

    'train': BASE_DIR / 'sample_temporal_train_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv',
    'val': BASE_DIR / 'sample_temporal_val_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv',
    'test': BASE_DIR / 'sample_temporal_test_hist2000_2022_target2023_2024_thr1_neg1_random_isolates.csv',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos requeridos para P9:\n' + '\n'.join(missing))

print(f'Dispositivo: {DEVICE}')
print(f'RUN_FINAL_TEST={RUN_FINAL_TEST}')
print(f'Semillas: {SEEDS[0]}..{SEEDS[-1]} ({len(SEEDS)} total)')


Dispositivo: cpu
RUN_FINAL_TEST=True
Semillas: 41..72 (32 total)


In [7]:
# Utilidades de grafo y features de nodo historicas


In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def node_type(G, node):
    return str(G.nodes[node].get('tipo', '')).strip().lower()


def is_article_node(G, node):
    return node_type(G, node) == 'articulo'


def is_author_node(G, node):
    return node_type(G, node) == 'autor'


def coauthorship_counts(G):
    counts = Counter()
    for article in G.nodes:
        if not is_article_node(G, article):
            continue
        authors = sorted({a for a in G.neighbors(article) if is_author_node(G, a)}, key=str)
        for u, v in combinations(authors, 2):
            pair = tuple(sorted((str(u), str(v))))
            counts[pair] += 1
    return counts


def build_author_graph(G_full, threshold=1):
    authors = sorted([str(n) for n in G_full.nodes if is_author_node(G_full, n)], key=str)
    H = nx.Graph()
    H.add_nodes_from(authors)
    counts = coauthorship_counts(G_full)
    for (u, v), n in counts.items():
        if n >= threshold:
            H.add_edge(u, v, weight=n)
    return H


def count_neighbor_type(G_full, author, tipo):
    if author not in G_full:
        return 0
    return sum(1 for n in G_full.neighbors(author) if node_type(G_full, n) == tipo)


def safe_core_numbers(H):
    if H.number_of_edges() == 0:
        return {n: 0 for n in H.nodes}
    try:
        return nx.core_number(H)
    except Exception:
        return {n: 0 for n in H.nodes}


def build_node_feature_frame(G_full, H):
    """
    Features de nodo calculadas EXCLUSIVAMENTE con el snapshot historico.
    No contienen ninguna etiqueta futura.
    """
    degree = dict(H.degree())
    clustering = nx.clustering(H)
    core = safe_core_numbers(H)
    try:
        pagerank = nx.pagerank(H, alpha=0.85, max_iter=500)
    except Exception:
        pagerank = {n: 0.0 for n in H.nodes}

    rows = []
    for a in H.nodes:
        rows.append({
            'author': str(a),
            'log_degree': np.log1p(float(degree.get(a, 0))),
            'clustering': float(clustering.get(a, 0.0)),
            'pagerank': float(pagerank.get(a, 0.0)),
            'log_core': np.log1p(float(core.get(a, 0))),
            'log_papers': np.log1p(float(count_neighbor_type(G_full, a, 'articulo'))),
            'log_keywords': np.log1p(float(count_neighbor_type(G_full, a, 'keyword'))),
            'log_affiliations': np.log1p(float(count_neighbor_type(G_full, a, 'afiliacion'))),
        })
    return pd.DataFrame(rows).set_index('author')


def graph_to_pyg(H, feature_df, scaler=None, fit_scaler=False):
    nodes = sorted([str(n) for n in H.nodes], key=str)
    node_to_idx = {n: i for i, n in enumerate(nodes)}

    X = feature_df.reindex(nodes).fillna(0.0).to_numpy(dtype=np.float32)
    if fit_scaler:
        scaler = StandardScaler()
        X = scaler.fit_transform(X).astype(np.float32)
    else:
        if scaler is None:
            raise ValueError('Se requiere scaler cuando fit_scaler=False')
        X = scaler.transform(X).astype(np.float32)

    edges = []
    for u, v in H.edges:
        i, j = node_to_idx[str(u)], node_to_idx[str(v)]
        edges.append((i, j))
        edges.append((j, i))
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    data = Data(x=torch.tensor(X, dtype=torch.float32), edge_index=edge_index)
    return data, node_to_idx, scaler


def load_pair_csv(path):
    df = pd.read_csv(path)
    required = {'source', 'target', 'connected'}
    miss = required - set(df.columns)
    if miss:
        raise ValueError(f'{path.name}: faltan columnas {sorted(miss)}')
    out = df[['source', 'target', 'connected']].copy()
    out['source'] = out['source'].astype(str)
    out['target'] = out['target'].astype(str)
    out['connected'] = out['connected'].astype(int)
    return out


def pair_tensors(df, node_to_idx, device):
    missing = sorted(({*df['source'], *df['target']} - set(node_to_idx)))
    if missing:
        raise ValueError(
            f'Hay {len(missing)} autores de pares que no existen en el snapshot historico. '
            f'Ejemplos: {missing[:5]}'
        )
    src = torch.tensor([node_to_idx[x] for x in df['source']], dtype=torch.long, device=device)
    dst = torch.tensor([node_to_idx[x] for x in df['target']], dtype=torch.long, device=device)
    y = torch.tensor(df['connected'].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    return src, dst, y


def verify_no_target_edges(H, pairs, split_name):
    positives = pairs[pairs['connected'] == 1]
    present = [(u, v) for u, v in positives[['source', 'target']].itertuples(index=False, name=None)
               if H.has_edge(str(u), str(v))]
    if present:
        raise AssertionError(
            f'{split_name}: {len(present)} aristas positivas objetivo ya estan presentes '
            'en el grafo usado para message passing.'
        )


In [9]:
# Modelos


In [10]:
class GNNEncoder(nn.Module):
    def __init__(self, model_name, in_dim, hidden_dim=32, out_dim=16, dropout=0.2):
        super().__init__()
        self.model_name = model_name
        self.dropout = dropout

        if model_name == 'GCN':
            self.conv1 = GCNConv(in_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, out_dim)
        elif model_name == 'GAT':
            heads = 4
            self.conv1 = GATConv(in_dim, hidden_dim // heads, heads=heads, dropout=dropout)
            self.conv2 = GATConv(hidden_dim, out_dim, heads=1, concat=False, dropout=dropout)
        elif model_name == 'GraphSAGE':
            self.conv1 = SAGEConv(in_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, out_dim)
        elif model_name == 'GIN':
            mlp1 = nn.Sequential(
                nn.Linear(in_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            mlp2 = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, out_dim)
            )
            self.conv1 = GINConv(mlp1, train_eps=True)
            self.conv2 = GINConv(mlp2, train_eps=True)
        else:
            raise ValueError(f'Modelo no reconocido: {model_name}')

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class PairDecoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim=32, dropout=0.2):
        super().__init__()
        # Simetrico respecto a u,v: [z_u*z_v, |z_u-z_v|]
        self.net = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, z, src, dst):
        zu, zv = z[src], z[dst]
        pair = torch.cat([zu * zv, torch.abs(zu - zv)], dim=1)
        return self.net(pair).squeeze(-1)


class LinkPredictor(nn.Module):
    def __init__(self, model_name, in_dim):
        super().__init__()
        self.encoder = GNNEncoder(model_name, in_dim, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.decoder = PairDecoder(EMBED_DIM, HIDDEN_DIM, DROPOUT)

    def encode(self, data):
        return self.encoder(data.x, data.edge_index)

    def score_pairs(self, z, src, dst):
        return self.decoder(z, src, dst)


def compute_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    out = {
        'auc_roc': np.nan,
        'auc_pr': np.nan,
        'f1': f1_score(y_true, pred, zero_division=0),
        'accuracy': accuracy_score(y_true, pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
    }
    if len(np.unique(y_true)) == 2:
        out['auc_roc'] = roc_auc_score(y_true, prob)
        out['auc_pr'] = average_precision_score(y_true, prob)
    return out


def predict(model, data, pair_df, node_to_idx):
    model.eval()
    data = data.to(DEVICE)
    src, dst, y = pair_tensors(pair_df, node_to_idx, DEVICE)
    with torch.no_grad():
        z = model.encode(data)
        logits = model.score_pairs(z, src, dst)
        prob = torch.sigmoid(logits).cpu().numpy()
    return y.cpu().numpy().astype(int), prob


def train_with_validation(model_name, seed, data_train_graph, train_df, val_df, node_to_idx):
    set_seed(seed)
    model = LinkPredictor(model_name, data_train_graph.num_node_features).to(DEVICE)
    data = data_train_graph.to(DEVICE)
    train_src, train_dst, train_y = pair_tensors(train_df, node_to_idx, DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    best_state = None
    best_val_pr = -np.inf
    best_epoch = 1
    wait = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        z = model.encode(data)
        logits = model.score_pairs(z, train_src, train_dst)
        loss = criterion(logits, train_y)
        loss.backward()
        optimizer.step()

        yv, pv = predict(model, data_train_graph, val_df, node_to_idx)
        val_pr = average_precision_score(yv, pv) if len(np.unique(yv)) == 2 else np.nan
        score = -np.inf if np.isnan(val_pr) else float(val_pr)

        if score > best_val_pr + MIN_DELTA:
            best_val_pr = score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    yv, pv = predict(model, data_train_graph, val_df, node_to_idx)
    metrics = compute_metrics(yv, pv)
    return best_epoch, metrics


def train_fixed_epochs(model_name, seed, data_train_graph, dev_df, node_to_idx, epochs):
    # Semilla distinta pero reproducible para el ajuste final con development completo.
    set_seed(seed + 10000)
    model = LinkPredictor(model_name, data_train_graph.num_node_features).to(DEVICE)
    data = data_train_graph.to(DEVICE)
    src, dst, y = pair_tensors(dev_df, node_to_idx, DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for _ in range(max(1, int(epochs))):
        optimizer.zero_grad()
        z = model.encode(data)
        logits = model.score_pairs(z, src, dst)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    return model


In [11]:
# Carga y auditoria temporal


In [12]:
print('\nCargando grafos historicos...')
G_2000_2020 = nx.read_gml(FILES['full_2000_2020'])
G_2021_2022 = nx.read_gml(FILES['full_2021_2022'])
G_2000_2022 = nx.compose(G_2000_2020, G_2021_2022)

H_2020 = build_author_graph(G_2000_2020, threshold=THRESHOLD)
H_2022 = build_author_graph(G_2000_2022, threshold=THRESHOLD)

print(f'H_2020: nodos={H_2020.number_of_nodes():,}, aristas={H_2020.number_of_edges():,}')
print(f'H_2022: nodos={H_2022.number_of_nodes():,}, aristas={H_2022.number_of_edges():,}')

train_df = load_pair_csv(FILES['train'])
val_df = load_pair_csv(FILES['val'])
test_df = load_pair_csv(FILES['test'])
dev_df = pd.concat([train_df, val_df], ignore_index=True)

verify_no_target_edges(H_2020, train_df, 'train')
verify_no_target_edges(H_2020, val_df, 'validation')
verify_no_target_edges(H_2022, test_df, 'test')

print('\nMuestras P4 reutilizadas:')
for name, df in [('train', train_df), ('val', val_df), ('development', dev_df), ('test', test_df)]:
    vc = df['connected'].value_counts().to_dict()
    print(f'{name}: n={len(df)}, positivos={vc.get(1, 0)}, negativos={vc.get(0, 0)}')

# Features de nodo historicas y escalado fit SOLO en 2000-2020.
feat_2020 = build_node_feature_frame(G_2000_2020, H_2020)
feat_2022 = build_node_feature_frame(G_2000_2022, H_2022)

data_2020, map_2020, scaler = graph_to_pyg(H_2020, feat_2020, fit_scaler=True)
data_2022, map_2022, _ = graph_to_pyg(H_2022, feat_2022, scaler=scaler, fit_scaler=False)

# Confirmar que los pares pertenecen al snapshot que les corresponde.
pair_tensors(dev_df, map_2020, torch.device('cpu'))
pair_tensors(test_df, map_2022, torch.device('cpu'))



Cargando grafos historicos...
H_2020: nodos=526, aristas=904
H_2022: nodos=1,493, aristas=3,060

Muestras P4 reutilizadas:
train: n=27, positivos=14, negativos=13
val: n=7, positivos=3, negativos=4
development: n=34, positivos=17, negativos=17
test: n=112, positivos=56, negativos=56


(tensor([  93,  588,  292,  318,   44,  504,   62,  887,  258,  118,  107,  318,
          260,  261,  709, 1292,  687,  598,  509,  626,   93,  566,   93,  353,
          302,  201, 1037,  215,  761,  509,  458, 1063,  566,  392,  260,   84,
          197,  412,  626,   93, 1179,  502,  783,  392,  765,  412,  509,  515,
          719,  483, 1396,  274, 1024,  193,  183,  946,  292,  738,  284, 1037,
           93, 1063,  509,  318,  460,  523,  282,  634,  774,  512,  573,  483,
           93,  400,  765,  647,  140,  514,  442,  193,  107,  201,   93,  412,
          887,   93,  140, 1206,  768,  237,  166,  457,   84,  193,  193,  687,
          690,  313,  483,  555,  709,   44,  504,  763,  566,  258,  237, 1079,
          483,  318,  690, 1339]),
 tensor([ 828, 1448,  910,  496,  283, 1235,  458, 1139,  260,  392,  947,  392,
          282,  805,  914, 1325,  783,  805,  523, 1019,  774, 1443, 1161, 1021,
         1161,  460, 1076, 1308,  947,  749,  548, 1076, 1456,  496, 1397,

In [13]:
# 20 semillas: seleccion de epoca con validation, sin tocar test por defecto


In [14]:
val_rows = []
best_epochs = {}

for model_name in MODEL_NAMES:
    best_epochs[model_name] = {}
    print(f'\n=== {model_name} ===')
    for seed in SEEDS:
        best_epoch, metrics = train_with_validation(
            model_name, seed, data_2020, train_df, val_df, map_2020
        )
        best_epochs[model_name][str(seed)] = int(best_epoch)
        row = {'model': model_name, 'seed': seed, 'best_epoch': best_epoch, **metrics}
        val_rows.append(row)
        print(
            f'seed={seed:02d} epoch={best_epoch:3d} '
            f'val_PR={metrics["auc_pr"]:.4f} val_ROC={metrics["auc_roc"]:.4f} '
            f'F1={metrics["f1"]:.4f}'
        )

val_runs = pd.DataFrame(val_rows)
val_runs.to_csv(OUT_DIR / 'P9_GNN_validation_runs.csv', index=False)

summary_cols = ['auc_roc', 'auc_pr', 'f1', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'best_epoch']
val_summary = val_runs.groupby('model')[summary_cols].agg(['mean', 'std', 'median', 'min', 'max'])
val_summary.columns = ['_'.join(c) for c in val_summary.columns]
val_summary = val_summary.reset_index()
val_summary.to_csv(OUT_DIR / 'P9_GNN_validation_summary.csv', index=False)

with open(OUT_DIR / 'P9_GNN_best_epochs.json', 'w', encoding='utf-8') as f:
    json.dump(best_epochs, f, indent=2)

print('\nResumen de VALIDACION (20 semillas):')
print(val_summary[['model', 'auc_pr_mean', 'auc_pr_std', 'auc_roc_mean', 'f1_mean', 'best_epoch_median']].to_string(index=False))



=== GCN ===
seed=41 epoch=  1 val_PR=0.6095 val_ROC=0.4167 F1=0.6000
seed=42 epoch=  1 val_PR=0.8056 val_ROC=0.8333 F1=0.6000
seed=43 epoch= 43 val_PR=0.6984 val_ROC=0.5833 F1=0.4000
seed=44 epoch= 25 val_PR=0.8095 val_ROC=0.6667 F1=0.6667
seed=45 epoch=  1 val_PR=0.8095 val_ROC=0.6667 F1=0.6000
seed=46 epoch=  2 val_PR=0.8095 val_ROC=0.6667 F1=0.0000
seed=47 epoch=  1 val_PR=0.6333 val_ROC=0.5000 F1=0.6000
seed=48 epoch=  3 val_PR=0.7222 val_ROC=0.6667 F1=0.6000
seed=49 epoch= 36 val_PR=0.5556 val_ROC=0.5833 F1=0.5714
seed=50 epoch=  4 val_PR=0.6667 val_ROC=0.5833 F1=0.6000
seed=51 epoch= 52 val_PR=0.8095 val_ROC=0.6667 F1=0.6667
seed=52 epoch= 22 val_PR=0.6984 val_ROC=0.5833 F1=0.5000
seed=53 epoch=  3 val_PR=0.7222 val_ROC=0.6667 F1=0.5000
seed=54 epoch= 14 val_PR=0.6667 val_ROC=0.5833 F1=0.5000
seed=55 epoch= 14 val_PR=0.8095 val_ROC=0.6667 F1=0.5000
seed=56 epoch= 25 val_PR=0.8095 val_ROC=0.6667 F1=0.5714
seed=57 epoch= 16 val_PR=0.6095 val_ROC=0.4167 F1=0.5000
seed=58 epoch= 18 

In [15]:
# Auditoria


In [16]:
audit = pd.DataFrame([
    {
        'run_final_test': RUN_FINAL_TEST,
        'threshold': THRESHOLD,
        'n_seeds': N_SEEDS,
        'models': ','.join(MODEL_NAMES),
        'message_passing_train_history': '2000-2020',
        'development_label_horizon': '2021-2022',
        'message_passing_test_history': '2000-2022',
        'test_label_horizon': '2023-2024',
        'future_edges_used_in_message_passing': False,
        'future_labels_used_as_node_features': False,
        'node_feature_scaler_fit_history': '2000-2020 only',
        'train_n': len(train_df),
        'val_n': len(val_df),
        'development_n': len(dev_df),
        'test_n': len(test_df),
        'test_used_for_epoch_selection': False,
        'decoder': 'symmetric MLP on elementwise-product and absolute-difference embeddings',
        'node_features': 'log_degree,clustering,pagerank,log_core,log_papers,log_keywords,log_affiliations',
    }
])
audit.to_csv(OUT_DIR / 'P9_GNN_audit.csv', index=False)


In [17]:
# Test final opcional


In [18]:
if RUN_FINAL_TEST:
    print('\n*** TEST FINAL ABIERTO: 2023-2024 ***')
    test_rows = []
    pred_rows = []

    for model_name in MODEL_NAMES:
        print(f'\n--- {model_name}: test final ---')
        for seed in SEEDS:
            epoch = best_epochs[model_name][str(seed)]
            model = train_fixed_epochs(
                model_name, seed, data_2020, dev_df, map_2020, epoch
            )
            yt, pt = predict(model, data_2022, test_df, map_2022)
            metrics = compute_metrics(yt, pt)
            test_rows.append({
                'model': model_name,
                'seed': seed,
                'selected_epoch': epoch,
                **metrics
            })

            local = test_df.copy()
            local['model'] = model_name
            local['seed'] = seed
            local['probability'] = pt
            local['prediction_0_5'] = (pt >= 0.5).astype(int)
            pred_rows.append(local)

            print(
                f'seed={seed:02d} PR={metrics["auc_pr"]:.4f} '
                f'ROC={metrics["auc_roc"]:.4f} F1={metrics["f1"]:.4f}'
            )

    test_runs = pd.DataFrame(test_rows)
    test_runs.to_csv(OUT_DIR / 'P9_GNN_FINAL_test_runs.csv', index=False)
    pd.concat(pred_rows, ignore_index=True).to_csv(
        OUT_DIR / 'P9_GNN_FINAL_test_predictions.csv', index=False
    )

    metric_cols = ['auc_roc', 'auc_pr', 'f1', 'accuracy', 'balanced_accuracy', 'precision', 'recall']
    test_summary = test_runs.groupby('model')[metric_cols].agg(['mean', 'std', 'median', 'min', 'max'])
    test_summary.columns = ['_'.join(c) for c in test_summary.columns]
    test_summary = test_summary.reset_index()
    test_summary.to_csv(OUT_DIR / 'P9_GNN_FINAL_test_summary.csv', index=False)

    print('\nResumen TEST FINAL:')
    print(test_summary[['model', 'auc_pr_mean', 'auc_pr_std', 'auc_roc_mean', 'f1_mean']].to_string(index=False))
else:
    print('\nRUN_FINAL_TEST=False: el test 2023-2024 NO fue evaluado.')
    print('Revise primero P9_GNN_validation_summary.csv, P9_GNN_validation_runs.csv y P9_GNN_audit.csv.')

print(f'\nArchivos guardados en: {OUT_DIR}')



*** TEST FINAL ABIERTO: 2023-2024 ***

--- GCN: test final ---
seed=41 PR=0.5838 ROC=0.5717 F1=0.6538
seed=42 PR=0.5366 ROC=0.5210 F1=0.0000
seed=43 PR=0.5759 ROC=0.5392 F1=0.4250
seed=44 PR=0.5780 ROC=0.5625 F1=0.2895
seed=45 PR=0.4880 ROC=0.4837 F1=0.0000
seed=46 PR=0.6113 ROC=0.5765 F1=0.6584
seed=47 PR=0.4731 ROC=0.4490 F1=0.6667
seed=48 PR=0.5120 ROC=0.5096 F1=0.5915
seed=49 PR=0.5846 ROC=0.5466 F1=0.3797
seed=50 PR=0.4979 ROC=0.4452 F1=0.5867
seed=51 PR=0.4297 ROC=0.3243 F1=0.0952
seed=52 PR=0.5958 ROC=0.5143 F1=0.1000
seed=53 PR=0.6412 ROC=0.6454 F1=0.6612
seed=54 PR=0.5492 ROC=0.4943 F1=0.3099
seed=55 PR=0.6660 ROC=0.6665 F1=0.4359
seed=56 PR=0.4853 ROC=0.4235 F1=0.1765
seed=57 PR=0.6321 ROC=0.6732 F1=0.6095
seed=58 PR=0.5204 ROC=0.4732 F1=0.2466
seed=59 PR=0.5304 ROC=0.4499 F1=0.1587
seed=60 PR=0.4904 ROC=0.4595 F1=0.2750
seed=61 PR=0.4905 ROC=0.4805 F1=0.4082
seed=62 PR=0.4537 ROC=0.4222 F1=0.1667
seed=63 PR=0.5561 ROC=0.5890 F1=0.5968
seed=64 PR=0.5547 ROC=0.5061 F1=0.3514
